In [20]:
import torch 
import os
import numpy as np
import torch.nn.functional as F
import pickle

from torch.utils.data import Dataset, random_split
from torch import nn
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [21]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [22]:
# download dataset and split train and test set

cf10_training_data = datasets.CIFAR10(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 

cf10_test_data = datasets.CIFAR10(root = 'Data', train = False, download= True, transform = transforms.ToTensor())


cf100_training_data = datasets.CIFAR100(root = 'data', train = True, download= True, transform= transforms.ToTensor()) 
cf100_test_data = datasets.CIFAR100(root = 'Data', train = False, download= True, transform = transforms.ToTensor())

In [23]:
base_folder = cf100_training_data.base_folder
root = cf100_training_data.root
file_path = os.path.join(root, base_folder, 'train')

with open(file_path, 'rb')as f:
    entry =pickle.load(f, encoding = 'latin1')
    cf100_training_data.targets = entry['coarse_labels']

file_path_test = os.path.join(root, base_folder, 'test')
with open(file_path_test, 'rb') as f:
    entry_test = pickle.load(f, encoding='latin1')
    cf100_test_data.targets = entry_test['coarse_labels']

C:\Users\sanja\AppData\Local\Temp\ipykernel_28392\2307198707.py:6: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry =pickle.load(f, encoding = 'latin1')
C:\Users\sanja\AppData\Local\Temp\ipykernel_28392\2307198707.py:11: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry_test = pickle.load(f, encoding='latin1')


In [24]:
#Splitting Logic
train_size = 40000
val_size = 10000

# print(len(cf100_training_data))

cf10_train_subset, cf10_val_subset = random_split(cf10_training_data, [train_size,val_size])
#cf100_val_data = random_split(cf100_training_data, [len(cf100_training_data)*0.8, len(cf100_training_data)*0.2])

In [25]:
print(1)
training_loader = torch.utils.data.DataLoader(cf10_train_subset, batch_size=32, shuffle=True)
validation_loader = torch.utils.data.DataLoader(cf10_val_subset, batch_size=32, shuffle=False)
#testing_loader = torch.utils.data.DataLoader(cf10_test_data, batch_size=32, shuffle=False)

1


In [26]:
class LesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = X.flatten(1)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X
        
model_1 = LesNet().to(device)

In [27]:
# model 2

class Model2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)

        self.fc1 = nn.Linear(5*5*16, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        #add dropuot 
        #self.dropout = nn.Dropout2d(p = 0.3) # 30% chance to be 0

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, (2,2))
        X = F.dropout2d(X, p = 0.3)  #add dropuot 

        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, (2,2))
        X = F.dropout2d(X, p = 0.3)  #add dropuot 
        
        X = X.flatten(1)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X

model_2 = Model2().to(device)

In [28]:
# model 3

class Model3(nn.Module): #Batch Normalization
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.bn1 = nn.BatchNorm2d(6)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)
        self.bn2 = nn.BatchNorm2d(16)
        

        self.fc1 = nn.Linear(5*5*16, 120)
        self.bn3 = nn.BatchNorm1d(120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

      

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.bn1(self.conv1(X)))
        X = F.max_pool2d(X, (2,2))
        

        X = F.relu(self.bn2(self.conv2(X)))
        X = F.max_pool2d(X, (2,2))
       

        X = X.flatten(1)

        X = F.relu(self.bn3(self.fc1(X)))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X

model_3 = Model3().to(device)

In [29]:
#Train for each epoch
def train_one_epoch(model, loader,optimizer, criterion):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for inputs, label in loader:
        if torch.cuda.is_available():
                inputs, label = inputs.cuda(), label.cuda()
                model.cuda()
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += label.size(0)
        correct += predicted.eq(label).sum().item()

    average_loss = running_loss / len(loader)
    
    accuracy = correct/ total * 100
    return average_loss, accuracy

In [30]:
#validate each epoch
def validate(model, loader, criterion):
    model.eval() #Set model to eval mode 
    running_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, label in loader:
            if torch.cuda.is_available():
                inputs, label = inputs.cuda(), label.cuda()
                model.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, label)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += label.size(0)
            correct += predicted.eq(label).sum().item()

        average_loss = running_loss / len(loader)
        accuracy = correct/ total * 100
        return average_loss, accuracy

            

In [31]:
def run_full_training_validate(model, train_loader, val_loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history = {"training_loss": [], "training_accuracy": [], "validation_loss": [], "validation_accuracy": []}
    print("---Training and Validation has started---")
    for epoch in range(epochs):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion)
        print(f"Epoch {epoch+1}, Training Loss : {train_loss}, Training Accuracy: {train_accuracy}")
        val_loss, val_accuracy = validate(model, val_loader, criterion)
        print(f"Epoch {epoch+1}, Validation Loss : {val_loss}, Validation Accuracy: {val_accuracy}")

        history['training_loss'].append(train_loss)
        history['training_accuracy'].append(train_accuracy)
        history['validation_loss'].append(val_loss)
        history['validation_accuracy'].append(val_accuracy)

    print('---Training and Validation has ended---')
    return history

In [32]:
history_1 = run_full_training_validate(model_1, training_loader, validation_loader)

---Training and Validation has started---
Epoch 1, Training Loss : 1.6530018844604493, Training Accuracy: 39.4675
Epoch 1, Validation Loss : 1.428681452434284, Validation Accuracy: 48.78
Epoch 2, Training Loss : 1.3523551055908203, Training Accuracy: 51.902499999999996
Epoch 2, Validation Loss : 1.2879442850621745, Validation Accuracy: 54.24
Epoch 3, Training Loss : 1.2308327414512634, Training Accuracy: 56.165
Epoch 3, Validation Loss : 1.2494149922182003, Validation Accuracy: 56.26
Epoch 4, Training Loss : 1.147763172721863, Training Accuracy: 59.6425
Epoch 4, Validation Loss : 1.2150389148404424, Validation Accuracy: 57.25
Epoch 5, Training Loss : 1.087750018596649, Training Accuracy: 61.82
Epoch 5, Validation Loss : 1.1964361564800763, Validation Accuracy: 58.230000000000004
Epoch 6, Training Loss : 1.034340649318695, Training Accuracy: 63.595
Epoch 6, Validation Loss : 1.141455891033331, Validation Accuracy: 59.730000000000004
Epoch 7, Training Loss : 0.9914605851173401, Training 

In [33]:
history_2 = run_full_training_validate(model_2, training_loader, validation_loader)

---Training and Validation has started---
Epoch 1, Training Loss : 1.96853203125, Training Accuracy: 28.017500000000002
Epoch 1, Validation Loss : 1.7881763472724646, Validation Accuracy: 36.0
Epoch 2, Training Loss : 1.7451543522834778, Training Accuracy: 37.145
Epoch 2, Validation Loss : 1.7146925541539542, Validation Accuracy: 38.36
Epoch 3, Training Loss : 1.6710021228790284, Training Accuracy: 39.8175
Epoch 3, Validation Loss : 1.6549228627841694, Validation Accuracy: 40.129999999999995
Epoch 4, Training Loss : 1.6145486159324647, Training Accuracy: 41.962500000000006
Epoch 4, Validation Loss : 1.6169047298522803, Validation Accuracy: 41.980000000000004
Epoch 5, Training Loss : 1.5800648454666137, Training Accuracy: 43.012499999999996
Epoch 5, Validation Loss : 1.605866086749604, Validation Accuracy: 42.74
Epoch 6, Training Loss : 1.550966116809845, Training Accuracy: 44.22
Epoch 6, Validation Loss : 1.5883610911262682, Validation Accuracy: 43.01
Epoch 7, Training Loss : 1.5288281

In [34]:
history_3 = run_full_training_validate(model_3, training_loader, validation_loader)

---Training and Validation has started---
Epoch 1, Training Loss : 1.5356624420642853, Training Accuracy: 44.529999999999994
Epoch 1, Validation Loss : 1.3124189801490345, Validation Accuracy: 53.13
Epoch 2, Training Loss : 1.2431073875427245, Training Accuracy: 55.877500000000005
Epoch 2, Validation Loss : 1.178609273685053, Validation Accuracy: 57.9
Epoch 3, Training Loss : 1.1257629666805267, Training Accuracy: 60.195
Epoch 3, Validation Loss : 1.1592277737852104, Validation Accuracy: 59.27
Epoch 4, Training Loss : 1.0388342891216278, Training Accuracy: 63.425
Epoch 4, Validation Loss : 1.0827789525635325, Validation Accuracy: 61.980000000000004
Epoch 5, Training Loss : 0.971846588563919, Training Accuracy: 65.605
Epoch 5, Validation Loss : 1.0574771921855572, Validation Accuracy: 62.62
Epoch 6, Training Loss : 0.9183104497671127, Training Accuracy: 67.5
Epoch 6, Validation Loss : 1.2042457130008613, Validation Accuracy: 59.31999999999999
Epoch 7, Training Loss : 0.8714212486505508,

In [35]:
#task 4  Current choice: model3(batch normalization) 
cf100_train_subset, cf100_val_subset = random_split(cf100_training_data, [train_size,val_size])

training_loader_cf100 = torch.utils.data.DataLoader(cf100_train_subset, batch_size=32, shuffle=True)
validation_loader_cf100 = torch.utils.data.DataLoader(cf100_val_subset, batch_size=32, shuffle=False)



In [36]:
class Model3_cf100(nn.Module): #it should be cf100 bruh
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,5,1)
        self.bn1 = nn.BatchNorm2d(6)
        self.conv2 = nn.Conv2d(6, 16, 5, 1)
        self.bn2 = nn.BatchNorm2d(16)
        

        self.fc1 = nn.Linear(5*5*16, 120)
        self.bn3 = nn.BatchNorm1d(120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 20)

      

        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu') #different relu

    def forward(self, X):
        X = F.relu(self.bn1(self.conv1(X)))
        X = F.max_pool2d(X, (2,2))
        

        X = F.relu(self.bn2(self.conv2(X)))
        X = F.max_pool2d(X, (2,2))
       

        X = X.flatten(1)

        X = F.relu(self.bn3(self.fc1(X)))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        return X

model_3_cf100 = Model3_cf100().to(device)

In [37]:
history_cf100_1 = run_full_training_validate(model_3_cf100, training_loader_cf100, validation_loader_cf100)

---Training and Validation has started---
Epoch 1, Training Loss : 2.436379815387726, Training Accuracy: 25.2975
Epoch 1, Validation Loss : 2.2170056215109537, Validation Accuracy: 31.5
Epoch 2, Training Loss : 2.1030247066497805, Training Accuracy: 34.73
Epoch 2, Validation Loss : 2.08506438945429, Validation Accuracy: 35.949999999999996
Epoch 3, Training Loss : 1.9611628576278686, Training Accuracy: 39.1125
Epoch 3, Validation Loss : 2.149110220491696, Validation Accuracy: 34.62
Epoch 4, Training Loss : 1.8665354072570801, Training Accuracy: 41.9975
Epoch 4, Validation Loss : 2.0189130382415965, Validation Accuracy: 37.64
Epoch 5, Training Loss : 1.7993598952293397, Training Accuracy: 43.7525
Epoch 5, Validation Loss : 1.9291681230258637, Validation Accuracy: 40.61
Epoch 6, Training Loss : 1.7413173263549804, Training Accuracy: 45.4125
Epoch 6, Validation Loss : 2.1149499039299573, Validation Accuracy: 36.71
Epoch 7, Training Loss : 1.6897816465377808, Training Accuracy: 46.92
Epoch 

In [38]:
def run_full_training_validate_2(model, train_loader, val_loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

    history = {"training_loss": [], "training_accuracy": [], "validation_loss": [], "validation_accuracy": []}
    print("---Training and Validation has started---")
    for epoch in range(epochs):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, criterion)
        print(f"Epoch {epoch+1}, Training Loss : {train_loss}, Training Accuracy: {train_accuracy}")
        val_loss, val_accuracy = validate(model, val_loader, criterion)
        print(f"Epoch {epoch+1}, Validation Loss : {val_loss}, Validation Accuracy: {val_accuracy}")

        history['training_loss'].append(train_loss)
        history['training_accuracy'].append(train_accuracy)
        history['validation_loss'].append(val_loss)
        history['validation_accuracy'].append(val_accuracy)

    print('---Training and Validation has ended---')
    return history

In [39]:
CIFAR10_pretrained = model_3_cf100
CIFAR10_pretrained.fc3 = nn.Linear(84, 10).to(device)
history_pretrained = run_full_training_validate_2(
    CIFAR10_pretrained, 
    training_loader,   
    validation_loader, 
    epochs=20
)

---Training and Validation has started---
Epoch 1, Training Loss : 1.4517362580776214, Training Accuracy: 47.245
Epoch 1, Validation Loss : 1.2372883000312902, Validation Accuracy: 54.459999999999994
Epoch 2, Training Loss : 1.2115554176807404, Training Accuracy: 56.330000000000005
Epoch 2, Validation Loss : 1.1604422281344478, Validation Accuracy: 57.699999999999996
Epoch 3, Training Loss : 1.125510125398636, Training Accuracy: 59.875
Epoch 3, Validation Loss : 1.1275793850993197, Validation Accuracy: 59.43000000000001
Epoch 4, Training Loss : 1.0591131491184234, Training Accuracy: 62.06
Epoch 4, Validation Loss : 1.1035371297083723, Validation Accuracy: 60.85
Epoch 5, Training Loss : 1.0102098958015442, Training Accuracy: 63.959999999999994
Epoch 5, Validation Loss : 1.080860465479354, Validation Accuracy: 61.550000000000004
Epoch 6, Training Loss : 0.9725000331878663, Training Accuracy: 65.3175
Epoch 6, Validation Loss : 1.0667903135759762, Validation Accuracy: 61.970000000000006
Ep

In [41]:
from sklearn.metrics import confusion_matrix, classification_report

def get_predictions(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_labels, all_preds

In [42]:
testing_loader = torch.utils.data.DataLoader(cf10_test_data, batch_size=32, shuffle=False)

In [43]:
labels_best, preds_best = get_predictions(model_3, testing_loader)
labels_pretrained, preds_pretrained = get_predictions(CIFAR10_pretrained, testing_loader)

In [ ]:
import seaborn as sns
cm = confusion_matrix(labels_pretrained, preds_pretrained)
plt.figure(figsize=(10,8))

              precision    recall  f1-score   support

           0       0.68      0.69      0.68      1000
           1       0.73      0.76      0.74      1000
           2       0.48      0.50      0.49      1000
           3       0.45      0.38      0.41      1000
           4       0.55      0.58      0.57      1000
           5       0.54      0.51      0.53      1000
           6       0.60      0.81      0.69      1000
           7       0.73      0.62      0.67      1000
           8       0.78      0.72      0.75      1000
           9       0.71      0.66      0.68      1000

    accuracy                           0.62     10000
   macro avg       0.63      0.62      0.62     10000
weighted avg       0.63      0.62      0.62     10000

